In [18]:
from liftlog import empty_log, log_workout, save_log, load_log
import pandas as pd
import plotly.graph_objects as go

#test
workout_log = load_log()
print(workout_log.shape)
print(workout_log["date"].dtype) 

(77, 6)
datetime64[us]


In [19]:
def get_exercise_history(df, exercise_name):
    """
    Filter the log down to one exercise, sorted by date, then by set_number within each day so sets stay in the order they were actually performed.
    """
    filtered = df[df["exercise_name"] == exercise_name].sort_values(["date", "set_number"]) #sort by date and set_number

    if filtered.empty: #if the filtered dataframe is empty, return None
        print(f"No data yet for {exercise_name}.")
        return filtered

    session_count = filtered["date"].nunique() #count the number of unique dates
    if session_count == 1: #if there is only one session, print a message
        print(f"Only one session logged for {exercise_name} — no trend yet.")

    return filtered

In [20]:
squat_history = get_exercise_history(workout_log, "Back Squat")
squat_history["date"].dtype
squat_history.head(8)

,date,exercise_name,set_number,reps,weight,notes
0,2026-05-24,Back Squat,1,5,165.0,NaN
1,2026-05-24,Back Squat,2,5,165.0,NaN
2,2026-05-24,Back Squat,3,5,165.0,NaN
3,2026-05-24,Back Squat,4,3,170.0,NaN
6,2026-05-28,Back Squat,1,5,170.0,NaN
7,2026-05-28,Back Squat,2,5,170.0,NaN
8,2026-05-28,Back Squat,3,5,170.0,NaN
9,2026-05-28,Back Squat,4,3,175.0,NaN


In [21]:
def plot_weight_history(df, title=None):
    """
    Plot weight over evenly spaced set index (row order).
    Expects columns: date, set_number, reps, weight, notes.
    """
    if df.empty:
        print("No data to plot.")
        return None

    # 1-based x so each logged set is one tick apart
    x = list(range(1, len(df) + 1))

    # Format hover fields (handles NaN notes and datetime dates)
    hover_dates = df["date"].dt.strftime("%Y-%m-%d")
    hover_notes = df["notes"].fillna("").astype(str)

    fig = go.Figure(
        go.Scatter(
            x=x,
            y=df["weight"],
            mode="lines+markers",
            marker=dict(size=8),
            customdata=list(zip(
                hover_dates,
                df["set_number"],
                df["reps"],
                df["weight"],
                hover_notes,
            )),
            hovertemplate=(
                "Date: %{customdata[0]}<br>"
                "Set #: %{customdata[1]}<br>"
                "Reps: %{customdata[2]}<br>"
                "Weight: %{customdata[3]} lbs<br>"
                "Notes: %{customdata[4]}<extra></extra>"
            ),
        )
    )

    if title is None and "exercise_name" in df.columns and not df["exercise_name"].empty:
        title = df["exercise_name"].iloc[0]

    fig.update_layout(
        title=title or "Weight history",
        xaxis_title="Set index (chronological order)",
        yaxis_title="Weight (lbs)",
        hovermode="closest",
    )

    return fig

In [22]:
squat_history_chart = plot_weight_history(squat_history)
squat_history_chart